# Project 2: wat Suriname aan de wereld verkoopt

**De vraag.** Wat verkoopt Suriname aan de wereld, en hoeveel daarvan komt uit edele metalen en stenen, waaronder goud? Uit welke soorten goederen bestond de export van Suriname in elk jaar van 2015 tot 2025, en welk deel van het totaal kwam uit hoofdstuk 71, edele metalen en stenen, waaronder goud?

**Voor wie het is.** Een ambtenaar van een ministerie die moet beoordelen hoe afhankelijk de exportinkomsten van Suriname zijn van één soort goederen. Op vrijdag geeft het team die ambtenaar een notitie van één pagina met twee grafieken en gecontroleerde getallen.

**De data.** De registratie van UN Comtrade van de export van Suriname naar de wereld, in Amerikaanse dollars, per hoofdstuk van het Geharmoniseerd Systeem, de code waarmee douanediensten goederen indelen. De tabel is op 2026-11-06 gedownload uit de openbare preview van UN Comtrade.

**Je team.** Vier rollen wisselen elke week: bouwer, datacontroleur, AI-controleur en presentator. Deze week controleert de datacontroleur dat elke optelling de juiste rijen gebruikt.

**Wat je vrijdag inlevert.** Dit notebook met elke CHECK geslaagd, de notitie voor de ambtenaar met de twee grafieken, en jullie AI-logboek.

## Laad de data

Deze cel krijg je cadeau. Hij leest de tabel uit de cursusmap, of bouwt hem opnieuw op uit UN Comtrade met één verzoek per jaar als het bestand er niet is.

Lees hem voor de vorm voordat je hem uitvoert: wat gaat erin, wat komt eruit, en waar begint het werk. Elke rij van de tabel is één hoofdstuk in één jaar, of het totaal van één jaar, en de kolom `level` zegt welke van de twee. De instelling `dtype` houdt `hs_code` als tekst, zodat hoofdstuk `03` zijn voorloopnul houdt.

In [ ]:
import json, time, urllib.request
import pandas as pd

BASE = "https://comtradeapi.un.org/public/v1/preview/C/A/HS"
NAMES = "https://comtradeapi.un.org/files/v1/app/reference/H3.json"

def download_exports(years=range(2015, 2026)):
    """Bouw de tabel opnieuw op uit UN Comtrade: Suriname (740), export (X), naar de wereld (0)."""
    names = {r["id"]: r["text"] for r in json.load(urllib.request.urlopen(NAMES))["results"]}
    rows = []
    for year in years:
        url = f"{BASE}?reporterCode=740&period={year}&flowCode=X&partnerCode=0&cmdCode=TOTAL,AG2"
        for r in json.load(urllib.request.urlopen(url))["data"]:
            code = r["cmdCode"]
            rows.append({"year": year,
                         "level": "total" if code == "TOTAL" else "chapter",
                         "hs_code": code,
                         "description": "All goods" if code == "TOTAL" else names[code].split(" - ", 1)[1].strip(),
                         "export_usd": r["primaryValue"]})
        time.sleep(1.5)                      # wacht tussen verzoeken aan de openbare server
    return pd.DataFrame(rows).sort_values(["year", "hs_code"]).reset_index(drop=True)

try:
    exports = pd.read_csv("https://santerre.ai/bigi-data/data/week02-suriname-exports.csv", dtype={"hs_code": str})
except FileNotFoundError:
    exports = download_exports()
exports

## TODO 1: beschrijf de tabel

Schrijf code die het aantal rijen, de kolomnamen, het eerste en laatste jaar, het aantal rijen per niveau en het aantal ontbrekende waarden per kolom afdrukt. Sla het aantal verschillende jaren op in `n_years` en het aantal rijen met het niveau `total` in `n_total_rows`.

In [ ]:
# TODO 1
n_years = None
n_total_rows = None

In [ ]:
# CHECK 1
assert n_years is not None and n_total_rows is not None, "n_years of n_total_rows is nog leeg"
assert n_years == exports["year"].nunique()
assert n_total_rows == (exports["level"] == "total").sum()
assert n_total_rows == n_years, "elk jaar hoort precies één totaalrij te hebben"
print("CHECK 1 geslaagd:", n_years, "jaren, één totaalrij per jaar")

## TODO 2: de delen en het geheel

Elk jaar heeft één rij met het niveau `total` en veel rijen met het niveau `chapter`. Sla in `chapter_sum` de som van de hoofdstukrijen per jaar op, en in `totals` de waarde van de totaalrij van elk jaar, beide met het jaar als index. Druk ze naast elkaar af en bekijk het verschil.

In [ ]:
# TODO 2
chapter_sum = None
totals = None

In [ ]:
# CHECK 2
assert chapter_sum is not None and totals is not None, "chapter_sum of totals is nog leeg"
assert list(chapter_sum.index) == list(totals.index) == sorted(exports["year"].unique())
assert (chapter_sum - totals).abs().max() < 1, "de hoofdstukken horen op te tellen tot het totaal, tot op een dollar"
print("CHECK 2 geslaagd: de hoofdstukken tellen op tot het totaal in alle", len(totals), "jaren")

## TODO 3: maak een grafiek van de totale export

Maak een lijngrafiek van de totale export per jaar, in miljoenen Amerikaanse dollars. De kolom `export_usd` is in Amerikaanse dollars, dus deel eerst door een miljoen. Geef de grafiek een titel, benoem beide assen met hun eenheden, en vermeld UN Comtrade als bron.

In [ ]:
# TODO 3

## TODO 4: wat Suriname verkoopt

Sla in `top5` een tabel op met de vijf hoofdstukken met de grootste export in 2025. Die bevat voor elk hoofdstuk de code, de omschrijving, de waarde en het aandeel in het totaal van 2025 in procenten. Sla hun codes, de grootste eerst, op in de lijst `top5_codes`.

Lees elke omschrijving helemaal. Sommige hoofdstuknamen zijn lang, en de eerste woorden zeggen niet altijd wat het belangrijkst is.

In [ ]:
# TODO 4
top5 = None
top5_codes = None

In [ ]:
# CHECK 4
ch2025 = exports[(exports["level"] == "chapter") & (exports["year"] == 2025)]
expected = list(ch2025.nlargest(5, "export_usd")["hs_code"])
assert top5_codes is not None, "top5_codes is nog leeg"
assert list(top5_codes) == expected, "vergelijk je lijst met de hoofdstukrijen van 2025, gesorteerd op waarde"
print("CHECK 4 geslaagd:", expected)

## TODO 5: het aandeel van edele metalen en stenen

Sla in `gold_share` het aandeel van hoofdstuk 71 in de totale export van elk jaar op, in procenten, met het jaar als index. Teken het als lijngrafiek met een titel, benoemde assen en de bron. Sla daarna in `low_year` en `high_year` de jaren op met het laagste en het hoogste aandeel.

Hoofdstuk 71 bevat edele metalen en stenen, waaronder goud. De tabel kan goud niet scheiden van de rest van het hoofdstuk.

In [ ]:
# TODO 5
gold_share = None
low_year = None
high_year = None

In [ ]:
# CHECK 5
t = exports[exports["level"] == "total"].set_index("year")["export_usd"]
c71 = exports[exports["hs_code"] == "71"].set_index("year")["export_usd"]
expected = 100 * c71 / t
assert gold_share is not None, "gold_share is nog leeg"
assert (gold_share.reindex(expected.index) - expected).abs().max() < 0.01
assert low_year == expected.idxmin() and high_year == expected.idxmax()
print("CHECK 5 geslaagd: hoofdstuk 71 was", round(expected[2025], 1), "% van de export in 2025; laagste", f"{int(low_year)},", "hoogste", int(high_year))

## TODO 6: lees de code van de AI en toets hem

Laat je AI-assistent de kolomnamen en de uitvoer van `exports.head()` zien, en vraag: *"Schrijf pandas-code die voor elk jaar het aandeel van hoofdstuk 71 in de totale export van Suriname afdrukt, in procenten."*

Voer de code nog niet uit. Plak hem in jullie AI-logboek en schrijf er drie antwoorden naast: welke rijen hij optelt om het totaal van elk jaar te krijgen, welke kolom hij door welke deelt, en in welke eenheid hij afdrukt. Voer hem daarna uit in een nieuwe cel en vergelijk zijn getallen met je antwoord op TODO 5. Als ze verschillen, zoek dan de regel die het verschil veroorzaakt.

Wat je AI ook schreef, meet één fout die elke lezer van deze tabel moet kunnen herkennen. Een programma dat alle rijen van een jaar optelt, de totaalrij inbegrepen, telt elke dollar twee keer. Sla in `share_if_every_row` het aandeel van hoofdstuk 71 per jaar op dat zo'n programma zou afdrukken.

Vraag de AI ten slotte, zonder hem de tabel te laten zien: *"Wat waren de drie grootste exportproducten van Suriname in 2025, en welk deel van de export was goud?"* Schrijf het antwoord in het AI-logboek en toets elk onderdeel aan TODO 4 en TODO 5.

In [ ]:
# TODO 6
share_if_every_row = None

In [ ]:
# CHECK 6
t = exports[exports["level"] == "total"].set_index("year")["export_usd"]
c71 = exports[exports["hs_code"] == "71"].set_index("year")["export_usd"]
expected = 100 * c71 / exports.groupby("year")["export_usd"].sum()
assert share_if_every_row is not None, "share_if_every_row is nog leeg"
assert (share_if_every_row.reindex(expected.index) - expected).abs().max() < 0.01
print("CHECK 6 geslaagd: alle rijen optellen geeft", round(expected[2025], 2), "% voor 2025 in plaats van", round(100 * c71[2025] / t[2025], 1), "%")

## TODO 7: de notitie voor de ambtenaar

Schrijf in een tekstcel hieronder een notitie van ongeveer 200 woorden voor een ambtenaar van een ministerie die moet beoordelen hoe afhankelijk de exportinkomsten van Suriname zijn van één soort goederen. Zet je twee grafieken eronder. Gebruik de getallen die je hebt berekend, noem de bron, zeg wat hoofdstuk 71 omvat, en zeg één ding dat de data de ambtenaar niet kan vertellen.